In [ ]:
import warnings
from pathlib import Path
from typing import Any, Dict, Iterable, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import ParameterGrid, train_test_split

warnings.filterwarnings("default")
warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")

RANDOM_STATE = 42
TEST_FRACTION = 1.0 / 3.0
VALIDATION_FRACTION_OF_REMAINDER = 0.5
EARLY_STOPPING_ROUNDS = 20
CLASSIFICATION_THRESHOLD = 0.5
TOP_N_GENES = 10
PLOT_DPI = 600

LTS_COLOR = "#80bcc8"
HTS_COLOR = "#d88f91"

SHAP_CMAP = LinearSegmentedColormap.from_list(
    "custom_shap_gradient",
    ["#39489f", "#39bbec", "#f9ed36", "#f38466", "#b81f25"],
)

plt.rcParams["figure.dpi"] = 150
plt.rcParams["savefig.dpi"] = PLOT_DPI
plt.rcParams["font.size"] = 11
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11


def get_code_dir() -> Path:
    """Return the script directory, or the current directory in a notebook."""
    try:
        return Path(__file__).resolve().parent
    except NameError:
        return Path.cwd()


def ensure_dir(path: Path) -> None:
    """Create a directory and any missing parent directories."""
    path.mkdir(parents=True, exist_ok=True)


def save_figure_svg(fig: plt.Figure, save_path: Path) -> None:
    """Save a Matplotlib figure as an SVG file and close it."""
    save_path = Path(save_path).with_suffix(".svg")
    fig.savefig(
        save_path,
        format="svg",
        bbox_inches="tight",
        facecolor="white",
        transparent=False,
    )
    plt.close(fig)
    print(f"[SAVED] {save_path}")


def validate_binary_labels(y: pd.Series) -> pd.Series:
    """Validate that labels contain only 0 and 1."""
    y_numeric = pd.to_numeric(y, errors="coerce")

    if y_numeric.isna().any():
        raise ValueError("The label column must contain only numeric values 0 and 1.")

    unique_labels = set(y_numeric.astype(int).unique())
    if not unique_labels.issubset({0, 1}) or len(unique_labels) != 2:
        raise ValueError("The label column must contain both classes: 0 for LTS and 1 for HTS.")

    return y_numeric.astype(int)


def load_dataset(data_path: Path) -> Tuple[pd.DataFrame, pd.Series, Sequence[str], Optional[str]]:
    """Load the expression matrix and return features, labels, and feature names."""
    if not data_path.exists():
        raise FileNotFoundError(f"Data file not found: {data_path}")

    df = pd.read_csv(data_path)

    if "label" not in df.columns:
        raise ValueError("The dataset must contain a 'label' column.")

    non_feature_columns = ["label"]
    identifier_column = None

    if df.columns[0] != "label" and not pd.api.types.is_numeric_dtype(df.iloc[:, 0]):
        identifier_column = str(df.columns[0])
        non_feature_columns.append(identifier_column)
        print(f"[INFO] Excluding identifier column from model features: {identifier_column}")

    feature_columns = [column for column in df.columns if column not in non_feature_columns]
    if not feature_columns:
        raise ValueError("No feature columns were found in the dataset.")

    X_numeric = df[feature_columns].apply(pd.to_numeric, errors="coerce")
    missing_count = int(X_numeric.isna().sum().sum())
    if missing_count > 0:
        print(
            f"[WARNING] {missing_count} missing or non-numeric feature values "
            "were replaced with 0.0."
        )

    X = X_numeric.fillna(0.0).astype(np.float32)
    y = validate_binary_labels(df["label"])

    return X, y, feature_columns, identifier_column


def save_confusion_matrix(
    cm: np.ndarray,
    class_names: Sequence[str],
    save_path: Path,
    title: str = "XGBoost Test Confusion Matrix",
) -> None:
    """Save a confusion matrix plot."""
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    image = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    fig.colorbar(image, ax=ax)

    ax.set(
        xticks=np.arange(cm.shape[1]),
        yticks=np.arange(cm.shape[0]),
        xticklabels=class_names,
        yticklabels=class_names,
        ylabel="True label",
        xlabel="Predicted label",
        title=title,
    )

    threshold = cm.max() / 2.0 if cm.max() > 0 else 0.5
    for row in range(cm.shape[0]):
        for column in range(cm.shape[1]):
            ax.text(
                column,
                row,
                format(cm[row, column], "d"),
                ha="center",
                va="center",
                color="white" if cm[row, column] > threshold else "black",
            )

    fig.tight_layout()
    save_figure_svg(fig, save_path)


def plot_roc_curve(y_true: Iterable[int], y_prob: np.ndarray, save_path: Path) -> None:
    """Save the receiver operating characteristic curve."""
    auc_value = roc_auc_score(y_true, y_prob)
    false_positive_rate, true_positive_rate, _ = roc_curve(y_true, y_prob)

    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    ax.plot(
        false_positive_rate,
        true_positive_rate,
        linewidth=2,
        label=f"ROC AUC = {auc_value:.4f}",
    )
    ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1.5)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("XGBoost Test ROC Curve")
    ax.legend(loc="lower right")

    fig.tight_layout()
    save_figure_svg(fig, save_path)


def plot_pr_curve(y_true: Iterable[int], y_prob: np.ndarray, save_path: Path) -> None:
    """Save the precision-recall curve."""
    average_precision = average_precision_score(y_true, y_prob)
    precision, recall, _ = precision_recall_curve(y_true, y_prob)

    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    ax.plot(recall, precision, linewidth=2, label=f"AP = {average_precision:.4f}")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title("XGBoost Test Precision-Recall Curve")
    ax.legend(loc="lower left")

    fig.tight_layout()
    save_figure_svg(fig, save_path)


def plot_test_score_distribution(
    y_true: Iterable[int],
    y_prob: np.ndarray,
    save_path: Path,
) -> None:
    """Save the predicted HTS-probability distribution for the test set."""
    y_array = np.asarray(y_true)

    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    ax.hist(
        y_prob[y_array == 0],
        bins=30,
        alpha=0.7,
        label="LTS (0)",
        edgecolor="black",
        color=LTS_COLOR,
    )
    ax.hist(
        y_prob[y_array == 1],
        bins=30,
        alpha=0.7,
        label="HTS (1)",
        edgecolor="black",
        color=HTS_COLOR,
    )
    ax.set_xlabel("Predicted probability of HTS")
    ax.set_ylabel("Cell count")
    ax.set_title("XGBoost Test Prediction Score Distribution")
    ax.legend()

    fig.tight_layout()
    save_figure_svg(fig, save_path)


def extract_binary_shap_values(shap_values_object: Any, n_features: int) -> np.ndarray:
    """Extract positive-class SHAP values across supported SHAP output formats."""
    values = (
        shap_values_object.values
        if hasattr(shap_values_object, "values")
        else shap_values_object
    )
    values = np.asarray(values)

    if values.ndim == 2:
        return values

    if values.ndim == 3:
        if values.shape[1] == n_features and values.shape[2] == 2:
            return values[:, :, 1]
        if values.shape[1] == 2 and values.shape[2] == n_features:
            return values[:, 1, :]

    raise ValueError(f"Unexpected SHAP output shape: {values.shape}")


def build_xgb_classifier(params: Dict[str, Any], early_stopping: bool) -> xgb.XGBClassifier:
    """Build an XGBoost binary classifier."""
    model_params = {
        "random_state": RANDOM_STATE,
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "n_jobs": -1,
        "tree_method": "hist",
        **params,
    }

    if early_stopping:
        model_params["early_stopping_rounds"] = EARLY_STOPPING_ROUNDS

    return xgb.XGBClassifier(**model_params)


def fit_with_early_stopping(
    model: xgb.XGBClassifier,
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_val: pd.DataFrame,
    y_val: pd.Series,
) -> xgb.XGBClassifier:
    """Fit an XGBoost model using validation AUC for boosting-round early stopping."""
    try:
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            verbose=False,
        )
    except TypeError:
        model.set_params(early_stopping_rounds=None)
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            verbose=False,
        )

    return model


def get_best_number_of_trees(model: xgb.XGBClassifier, default_value: int) -> int:
    """Return the number of boosting rounds selected by early stopping."""
    best_iteration = getattr(model, "best_iteration", None)
    if best_iteration is not None:
        return int(best_iteration) + 1

    best_ntree_limit = getattr(model, "best_ntree_limit", None)
    if best_ntree_limit is not None and best_ntree_limit > 0:
        return int(best_ntree_limit)

    return int(default_value)


def get_bar_colors(values: np.ndarray) -> Tuple[np.ndarray, Normalize]:
    """Map numeric values to colors for SHAP importance plots."""
    values = np.asarray(values, dtype=float)

    if values.size == 0:
        return np.empty((0, 4)), Normalize(vmin=0.0, vmax=1.0)

    minimum = float(values.min())
    maximum = float(values.max())

    if np.isclose(minimum, maximum):
        norm = Normalize(vmin=0.0, vmax=1.0)
        normalized_values = np.full(values.shape, 0.5, dtype=float)
    else:
        norm = Normalize(vmin=minimum, vmax=maximum)
        normalized_values = norm(values)

    return SHAP_CMAP(normalized_values), norm


def plot_top_shap_importance(
    shap_importance_df: pd.DataFrame,
    save_path: Path,
    top_n: int = TOP_N_GENES,
) -> None:
    """Save a horizontal bar plot of the top genes by mean absolute SHAP value."""
    plot_df = (
        shap_importance_df
        .sort_values("mean_abs_shap", ascending=False)
        .head(top_n)
        .iloc[::-1]
        .copy()
    )

    colors, norm = get_bar_colors(plot_df["mean_abs_shap"].to_numpy())

    fig, ax = plt.subplots(figsize=(7, max(4.5, plot_df.shape[0] * 0.42)))
    ax.barh(plot_df["gene"], plot_df["mean_abs_shap"], color=colors)
    ax.set_xlabel("Mean |SHAP value|")
    ax.set_ylabel("Gene")
    ax.set_title(f"Top {plot_df.shape[0]} XGBoost Genes by SHAP Importance")

    scalar_mappable = ScalarMappable(norm=norm, cmap=SHAP_CMAP)
    scalar_mappable.set_array([])
    colorbar = fig.colorbar(scalar_mappable, ax=ax)
    colorbar.set_label("Mean |SHAP value|")

    fig.tight_layout()
    save_figure_svg(fig, save_path)


def plot_top_shap_importance_by_class(
    shap_importance_df: pd.DataFrame,
    save_path: Path,
    top_n: int = TOP_N_GENES,
) -> None:
    """Compare class-specific mean absolute SHAP values for the top genes."""
    plot_df = (
        shap_importance_df
        .sort_values("mean_abs_shap", ascending=False)
        .head(top_n)
        .iloc[::-1]
        .copy()
    )

    y_positions = np.arange(plot_df.shape[0])
    bar_height = 0.38

    fig, ax = plt.subplots(figsize=(7, max(4.5, plot_df.shape[0] * 0.42)))
    ax.barh(
        y_positions - bar_height / 2,
        plot_df["mean_abs_shap_LTS"],
        height=bar_height,
        label="LTS (0)",
        color=LTS_COLOR,
    )
    ax.barh(
        y_positions + bar_height / 2,
        plot_df["mean_abs_shap_HTS"],
        height=bar_height,
        label="HTS (1)",
        color=HTS_COLOR,
    )
    ax.set_yticks(y_positions)
    ax.set_yticklabels(plot_df["gene"])
    ax.set_xlabel("Mean |SHAP value|")
    ax.set_ylabel("Gene")
    ax.set_title(f"Top {plot_df.shape[0]} SHAP Genes by Microglial State")
    ax.legend()

    fig.tight_layout()
    save_figure_svg(fig, save_path)


def run_final_model_shap(
    final_model: xgb.XGBClassifier,
    X_all: pd.DataFrame,
    y_all: pd.Series,
    feature_columns: Sequence[str],
    output_dir: Path,
    figure_dir: Path,
) -> None:
    """Calculate SHAP values for the refitted final model on the complete dataset."""
    import shap

    print("[INFO] Calculating SHAP values for the refitted final model...")

    explainer = shap.TreeExplainer(final_model)
    shap_values_object = explainer(X_all)
    shap_values = extract_binary_shap_values(
        shap_values_object,
        n_features=len(feature_columns),
    )

    if shap_values.shape != X_all.shape:
        raise ValueError(
            "The SHAP matrix shape does not match the feature matrix: "
            f"SHAP={shap_values.shape}, X={X_all.shape}."
        )

    y_array = y_all.to_numpy()
    lts_mask = y_array == 0
    hts_mask = y_array == 1

    shap_importance_df = pd.DataFrame(
        {
            "gene": feature_columns,
            "mean_abs_shap": np.abs(shap_values).mean(axis=0),
            "mean_shap": shap_values.mean(axis=0),
            "mean_abs_shap_LTS": np.abs(shap_values[lts_mask]).mean(axis=0),
            "mean_abs_shap_HTS": np.abs(shap_values[hts_mask]).mean(axis=0),
            "mean_shap_LTS": shap_values[lts_mask].mean(axis=0),
            "mean_shap_HTS": shap_values[hts_mask].mean(axis=0),
        }
    ).sort_values("mean_abs_shap", ascending=False)

    shap_importance_df.to_csv(
        output_dir / "xgboost_final_model_shap_gene_importance.csv",
        index=False,
    )
    shap_importance_df.head(TOP_N_GENES).to_csv(
        output_dir / f"top{TOP_N_GENES}_xgboost_final_model_shap_genes.csv",
        index=False,
    )

    shap.summary_plot(
        shap_values,
        X_all,
        feature_names=feature_columns,
        show=False,
        plot_type="dot",
        max_display=30,
    )
    summary_figure = plt.gcf()
    summary_figure.set_size_inches(10, 8)
    save_figure_svg(summary_figure, figure_dir / "xgboost_final_model_shap_summary.svg")

    plot_top_shap_importance(
        shap_importance_df,
        figure_dir / f"top{TOP_N_GENES}_xgboost_shap_importance.svg",
        top_n=TOP_N_GENES,
    )
    plot_top_shap_importance_by_class(
        shap_importance_df,
        figure_dir / f"top{TOP_N_GENES}_xgboost_shap_importance_by_state.svg",
        top_n=TOP_N_GENES,
    )

    print(
        f"[INFO] Saved the top {TOP_N_GENES} genes ranked by mean absolute SHAP value."
    )


def main() -> None:
    code_dir = get_code_dir()
    data_path = code_dir / "df_expr.csv"
    output_dir = code_dir / "xgboost_hts_lts_results"
    figure_dir = output_dir / "figures"

    ensure_dir(output_dir)
    ensure_dir(figure_dir)

    print(f"[INFO] Working directory: {code_dir}")
    print(f"[INFO] Reading dataset: {data_path}")

    X, y, feature_columns, identifier_column = load_dataset(data_path)

    print(f"[INFO] Feature matrix shape: {X.shape}")
    print(f"[INFO] HTS cells (label 1): {int((y == 1).sum())}")
    print(f"[INFO] LTS cells (label 0): {int((y == 0).sum())}")

    X_development, X_test, y_development, y_test = train_test_split(
        X,
        y,
        test_size=TEST_FRACTION,
        random_state=RANDOM_STATE,
        stratify=y,
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_development,
        y_development,
        test_size=VALIDATION_FRACTION_OF_REMAINDER,
        random_state=RANDOM_STATE,
        stratify=y_development,
    )

    print(f"[INFO] Training set size: {X_train.shape[0]}")
    print(f"[INFO] Validation set size: {X_val.shape[0]}")
    print(f"[INFO] Test set size: {X_test.shape[0]}")

    parameter_grid = {
        "n_estimators": [500],
        "max_depth": [3, 6],
        "learning_rate": [0.03, 0.1],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8],
        "gamma": [0],
        "scale_pos_weight": [1, 2],
    }

    parameter_combinations = list(ParameterGrid(parameter_grid))
    tuning_records = []
    best_validation_auc = -np.inf
    best_parameters = None
    best_number_of_trees = None

    print(
        f"[INFO] Evaluating {len(parameter_combinations)} XGBoost parameter combinations."
    )

    for trial_number, parameters in enumerate(parameter_combinations, start=1):
        candidate_model = build_xgb_classifier(parameters, early_stopping=True)
        candidate_model = fit_with_early_stopping(
            candidate_model,
            X_train,
            y_train,
            X_val,
            y_val,
        )

        validation_probability = candidate_model.predict_proba(X_val)[:, 1]
        validation_prediction = (
            validation_probability >= CLASSIFICATION_THRESHOLD
        ).astype(int)

        validation_auc = roc_auc_score(y_val, validation_probability)
        validation_accuracy = accuracy_score(y_val, validation_prediction)
        validation_f1 = f1_score(y_val, validation_prediction, zero_division=0)
        validation_precision = precision_score(
            y_val,
            validation_prediction,
            zero_division=0,
        )
        validation_recall = recall_score(
            y_val,
            validation_prediction,
            zero_division=0,
        )
        number_of_trees_used = get_best_number_of_trees(
            candidate_model,
            default_value=parameters["n_estimators"],
        )

        tuning_records.append(
            {
                "trial": trial_number,
                "max_depth": parameters["max_depth"],
                "learning_rate": parameters["learning_rate"],
                "n_estimators_max": parameters["n_estimators"],
                "n_estimators_used": number_of_trees_used,
                "subsample": parameters["subsample"],
                "colsample_bytree": parameters["colsample_bytree"],
                "gamma": parameters["gamma"],
                "scale_pos_weight": parameters["scale_pos_weight"],
                "validation_auc": validation_auc,
                "validation_accuracy": validation_accuracy,
                "validation_f1": validation_f1,
                "validation_precision": validation_precision,
                "validation_recall": validation_recall,
            }
        )

        print(
            f"[INFO] Trial {trial_number}/{len(parameter_combinations)} | "
            f"max_depth={parameters['max_depth']} | "
            f"learning_rate={parameters['learning_rate']} | "
            f"trees_used={number_of_trees_used} | "
            f"subsample={parameters['subsample']} | "
            f"colsample_bytree={parameters['colsample_bytree']} | "
            f"scale_pos_weight={parameters['scale_pos_weight']} | "
            f"validation_AUC={validation_auc:.4f}"
        )

        if validation_auc > best_validation_auc:
            best_validation_auc = validation_auc
            best_parameters = parameters.copy()
            best_number_of_trees = number_of_trees_used

    if best_parameters is None or best_number_of_trees is None:
        raise RuntimeError("No XGBoost model was fitted successfully.")

    tuning_df = pd.DataFrame(tuning_records).sort_values(
        "validation_auc",
        ascending=False,
    )
    tuning_df.to_csv(output_dir / "validation_tuning_results.csv", index=False)

    print(f"[INFO] Best validation AUC: {best_validation_auc:.4f}")
    print(f"[INFO] Best parameters: {best_parameters}")
    print(f"[INFO] Selected number of trees: {best_number_of_trees}")

    X_trainval = pd.concat([X_train, X_val], axis=0)
    y_trainval = pd.concat([y_train, y_val], axis=0)

    final_parameters = best_parameters.copy()
    final_parameters["n_estimators"] = best_number_of_trees
    final_model = build_xgb_classifier(final_parameters, early_stopping=False)

    print("[INFO] Refitting the final model on the combined training and validation sets...")
    final_model.fit(X_trainval, y_trainval, verbose=False)

    print("[INFO] Evaluating the final model on the independent test set...")
    test_probability = final_model.predict_proba(X_test)[:, 1]
    test_prediction = (test_probability >= CLASSIFICATION_THRESHOLD).astype(int)

    test_auc = roc_auc_score(y_test, test_probability)
    test_accuracy = accuracy_score(y_test, test_prediction)
    test_f1 = f1_score(y_test, test_prediction, zero_division=0)
    test_precision = precision_score(y_test, test_prediction, zero_division=0)
    test_recall = recall_score(y_test, test_prediction, zero_division=0)
    test_average_precision = average_precision_score(y_test, test_probability)

    metrics_df = pd.DataFrame(
        [
            {
                "best_validation_auc": best_validation_auc,
                "test_auc": test_auc,
                "test_accuracy": test_accuracy,
                "test_f1": test_f1,
                "test_precision": test_precision,
                "test_recall": test_recall,
                "test_average_precision": test_average_precision,
                "classification_threshold": CLASSIFICATION_THRESHOLD,
                "best_max_depth": final_parameters["max_depth"],
                "best_learning_rate": final_parameters["learning_rate"],
                "best_n_estimators": final_parameters["n_estimators"],
                "best_subsample": final_parameters["subsample"],
                "best_colsample_bytree": final_parameters["colsample_bytree"],
                "best_gamma": final_parameters["gamma"],
                "best_scale_pos_weight": final_parameters["scale_pos_weight"],
                "early_stopping_rounds": EARLY_STOPPING_ROUNDS,
            }
        ]
    )
    metrics_df.to_csv(output_dir / "test_metrics.csv", index=False)

    report = classification_report(
        y_test,
        test_prediction,
        target_names=["LTS", "HTS"],
        digits=4,
        zero_division=0,
    )
    with open(
        output_dir / "classification_report.txt",
        "w",
        encoding="utf-8",
    ) as report_file:
        report_file.write(report)

    prediction_df = pd.DataFrame(
        {
            "true_label": y_test.to_numpy(),
            "true_class": y_test.map({0: "LTS", 1: "HTS"}).to_numpy(),
            "predicted_label": test_prediction,
            "predicted_class": pd.Series(test_prediction).map(
                {0: "LTS", 1: "HTS"}
            ).to_numpy(),
            "predicted_probability_HTS": test_probability,
        }
    )
    prediction_df.to_csv(output_dir / "test_predictions.csv", index=False)

    confusion = confusion_matrix(y_test, test_prediction)
    save_confusion_matrix(
        confusion,
        ["LTS", "HTS"],
        figure_dir / "xgboost_test_confusion_matrix.svg",
    )
    plot_roc_curve(
        y_test,
        test_probability,
        figure_dir / "xgboost_test_roc_curve.svg",
    )
    plot_pr_curve(
        y_test,
        test_probability,
        figure_dir / "xgboost_test_precision_recall_curve.svg",
    )
    plot_test_score_distribution(
        y_test,
        test_probability,
        figure_dir / "xgboost_test_score_distribution.svg",
    )

    try:
        run_final_model_shap(
            final_model=final_model,
            X_all=X,
            y_all=y,
            feature_columns=feature_columns,
            output_dir=output_dir,
            figure_dir=figure_dir,
        )
    except Exception as error:
        raise RuntimeError(f"Final-model SHAP analysis failed: {error}") from error

    print("[RESULT] Analysis completed successfully.")
    print(f"[RESULT] Output directory: {output_dir}")
    print(f"[RESULT] Best validation AUC: {best_validation_auc:.4f}")
    print(f"[RESULT] Test AUC: {test_auc:.4f}")
    print(f"[RESULT] Test accuracy: {test_accuracy:.4f}")
    print(f"[RESULT] Test F1 score: {test_f1:.4f}")


if __name__ == "__main__":
    main()
